# Gemma 4 31B via OpenRouter

Quick setup to call `google/gemma-4-31b-it` through the OpenRouter API for testing combined stories.

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_KEY"],
)

MODEL = "google/gemma-4-31b-it"

In [8]:
system_prompt = """
You are a fiction generator producing short story excerpts for a research dataset.

## Task

You will be given:
- THREE emotions
- A MODE: either SIMULTANEOUS or SEQUENTIAL
- A SETTING seed, a POV, and a TENSE

Write ONE excerpt of 150-200 words that conveys all three emotions
according to the MODE.

## The hard rule

NEVER write the name of any assigned emotion, and NEVER write a direct
synonym of it. This is the single most important rule. A story that names
its emotion is worthless for this dataset and will be discarded.

Convey emotion ONLY through:
- actions and behaviors
- physical sensations and body language
- dialogue and tone of voice
- thoughts and internal reactions
- situational context and environmental description

Banned in every story, regardless of assigned emotions: "felt", "feeling",
"feelings", "emotion", "emotional".

Do not substitute a near-synonym for a banned word. If the emotion is
"furious", you may not write furious, angry, mad, enraged, livid, irate,
seething, or fuming. Write the clenched jaw instead.

Do not name the emotion in a simile or a metaphor either. "A wave of
sadness" is a violation. "Like someone who had just been told bad news"
is a violation if it names the state.

## MODE: SIMULTANEOUS

All three emotions are present throughout the whole excerpt. Do not put
them in separate paragraphs. Do not resolve one into another. The reader
should be able to point at almost any sentence and find at least two of
them operating at once.

Techniques that work:
- give the emotions different objects (one about a person, one about an event, one about the room)
- give them different layers (one displayed to others, one felt privately)
- give them different timescales (one ambient and long-running, one immediate)
- let the body do one thing while the thoughts do another
- Include an XML tag in the format of <emotion>(happy, sad, angry)</emotion> to denote the emotions that the generated story uses.

## MODE: SEQUENTIAL

The excerpt moves through the three emotions in the order given, in three
distinct phases of roughly equal length. Each phase should be clearly one
state. Mark each transition with a concrete external event — a door
opening, a phone lighting up, a line of dialogue, stepping outside — not
with a summary sentence about the character changing.

Make sure that the order of emotions in the story follows the order of emotions (Emotion 1, etc...) given in the user prompt.

Do not blend the phases. Phase 2 should not contain residue of phase 1. Generate XML tags in the format <emotion>happy</emotion> at each phase transition, INCLUDING at the beginning of the story. 

## Style requirements

- One continuous scene, no time skips beyond the transitions.
- Only generate stories in 3rd-person past tense.
- Concrete and specific. Name objects, sounds, textures, and small physical details. Avoid abstraction.
- No dream sequences, no framing devices, no narrator commenting on the story.
- Do not begin with the character waking up.
- Do not end with a summary line that explains what happened.

## Output format

Output ONLY the story text. No title, no preamble, no explanation, no
labels, no quotation marks around the whole thing, no notes about which
emotion appears where. Begin with the first word of the story. 
"""

In [10]:
import itertools

def generate_messages_template(emotions:list[str], n:int):
    '''
    Makes template for user prompt for generating stories using listed emotions,
    n//2 simultaneous and the other sequential
    '''
    assert len(emotions) == 3
    output = []
    for idx in range(n):
        order = idx % 6
        mode = "SIMULTANEOUS" if idx < n/2 else "SEQUENTIAL"
        perms = list(itertools.permutations(emotions))
        output.append(f"""
            Emotion 1: {perms[order][0]}
            Emotion 2: {perms[order][1]}
            Emotion 3: {perms[order][2]}
            Mode: {mode}
        """)
    return output
        
    

In [ ]:
user_prompt = generate_messages_template(["at ease","rejuvenated","blissful"],6)[0]
print(user_prompt)
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}],
)
print(response.choices[0].message.content)